# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata information
meta = dataset.metadata
print(f'Name: {getattr(meta, "name", None)}')
print(f'Description: {getattr(meta, "description", None)}')
print(f'Citation: {getattr(meta, "citeAs", None)}')
print(f'Published: {getattr(meta, "datePublished", None)}')

## 2. Data Overview
Review available record sets, fields, and their IDs.

A Croissant dataset consists of record sets. Each record set contains fields, which are mapped to columns in files. All are referenced by their `@id`. Let's list all record sets, their `@id`, and their fields.

In [ ]:
# List all record sets and their fields
print('Available record sets:')
for record_set in dataset.metadata.record_sets:
    print(f'  RecordSet name: {getattr(record_set, "name", "")}, @id: {record_set.id}')
    print('    Fields:')
    for field in getattr(record_set, 'fields', []):
        print(f'      - {getattr(field, "name", "")} (field @id: {field.id}) [column: {getattr(field, "column", None)}]')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this dataset, the main record set appears to contain protein abundances, peptides, and modifications. We'll operate on the *main* record set (`@id`) as determined in the previous cell.

In [ ]:
# Select the record sets of interest (by @id from above)
# For this dataset, let's collect all available record_set @ids:
record_set_ids = [r.id for r in dataset.metadata.record_sets]
print('Loading all record sets:')
print(record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if not records:
        print(f'No records found for record set: {rs_id}')
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Choose the main record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Choose the first available
    print(f'Using main record set for further analysis: {main_record_set_id}')
    print('Field (DataFrame column) names:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets contained records! Check dataset definition.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll work with numerical columns such as peptide count or abundance, filtering out proteins with very small counts, and normalizing values. Please replace `<NUMERIC_FIELD_COLUMN_NAME>` and `<GROUP_FIELD_COLUMN_NAME>` with the actual column names as detected above (hint: look for fields containing 'abundance', 'count', or 'MW' as likely numeric fields).

In [ ]:
# Set the field (column) names from previous listings
main_df = dataframes[main_record_set_id]

# Try to automatically detect a numeric field for example EDA (e.g., peptide count, MW, abundance)
numeric_candidates = [c for c in main_df.columns if main_df[c].dtype in [float, int] or pd.api.types.is_numeric_dtype(main_df[c])]
print('Detected numeric candidates:', numeric_candidates)
# Fallback: if no float/int, try to coerce first numeric-like column
if not numeric_candidates:
    for c in main_df.columns:
        try:
            main_df[c] = pd.to_numeric(main_df[c])
            numeric_candidates = [c]
            break
        except Exception:
            continue

if not numeric_candidates:
    print('No numeric fields detected. Please review available columns.')
else:
    numeric_field = numeric_candidates[0]
    threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field for these records
    normalized = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = normalized
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a field if categorical fields available
    # Try to find a groupable field (e.g., protein class/description/some sample id)
    group_candidates = [c for c in main_df.columns if main_df[c].dtype == object and main_df[c].nunique() < 25 and c != numeric_field]
    print('Detected possible groupable fields:', group_candidates)
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print('No appropriate categorical field found to group by.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and, if possible, show group-wise means. (If you run this notebook in Colab or Jupyter, figures will be displayed inline.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # Grouped barplot if possible
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        means = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        sns.barplot(data=means, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f'Mean {numeric_field} grouped by {group_field}')
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
This notebook demonstrates how to load, explore, and analyze a Croissant-based FAIR dataset using `mlcroissant`. We've:
- Inspected metadata
- Enumerated record sets and their field `@id`s
- Loaded tabular records into DataFrames
- Ran basic filtering, normalization, and grouping on numeric fields
- Created simple plots for EDA

### Next Steps
- Explore additional fields and relationships
- Connect with original manuscript or ontology for field interpretations
- Apply domain-specific analyses (e.g., protein family enrichment, cross-sample comparisons)

> Please consult https://mlcommons.org/croissant/ for more advanced data processing examples.